In [ ]:
from google.colab import drive, files
drive.mount('/content/drive')

In [8]:
!mkdir -p ~/.kaggle
!mv kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json
!kaggle datasets list

ref                                                             title                                                    size  lastUpdated                 downloadCount  voteCount  usabilityRating  
--------------------------------------------------------------  -------------------------------------------------  ----------  --------------------------  -------------  ---------  ---------------  
dmahajanbe23/bmw-global-automotive-sales                        BMW Global Automotive Sales                             55017  2026-02-22 18:18:38.170000          11199        226                1  
ssssws/chocolate-sales-dataset-2023-2024                        Chocolate Sales Dataset 2023 - 2024                  24420255  2026-03-07 04:58:02.387000           3544         67                1  
sahilislam007/spotify-user-behavior-and-pattern                 Spotify User Behavior And Pattern                     1045077  2026-03-12 07:13:26.360000            846         29                1  
rhyth

In [9]:
import os
!kaggle datasets download -d aliabdelmenam/rdd-2022
!unzip rdd-2022.zip -d /content/RDD2022
print(os.listdir('/content/RDD2022'))

Streaming output truncated to the last 5000 lines.
  inflating: /content/RDD2022/RDD_SPLIT/val/labels/Czech_000931.txt  
  inflating: /content/RDD2022/RDD_SPLIT/val/labels/Czech_000939.txt  
  inflating: /content/RDD2022/RDD_SPLIT/val/labels/Czech_000949.txt  
  inflating: /content/RDD2022/RDD_SPLIT/val/labels/Czech_000957.txt  
  inflating: /content/RDD2022/RDD_SPLIT/val/labels/Czech_000962.txt  
  inflating: /content/RDD2022/RDD_SPLIT/val/labels/Czech_000963.txt  
  inflating: /content/RDD2022/RDD_SPLIT/val/labels/Czech_000966.txt  
  inflating: /content/RDD2022/RDD_SPLIT/val/labels/Czech_000971.txt  
  inflating: /content/RDD2022/RDD_SPLIT/val/labels/Czech_000977.txt  
  inflating: /content/RDD2022/RDD_SPLIT/val/labels/Czech_000990.txt  
  inflating: /content/RDD2022/RDD_SPLIT/val/labels/Czech_000992.txt  
  inflating: /content/RDD2022/RDD_SPLIT/val/labels/Czech_000995.txt  
  inflating: /content/RDD2022/RDD_SPLIT/val/labels/Czech_000997.txt  
  inflating: /content/RDD2022/RDD_SPLIT

In [13]:
import os, shutil, random, glob
from collections import defaultdict

random.seed(42)

RDD_ROOT = '/content/RDD2022/RDD_SPLIT'
OUT_DIR  = '/content/dataset'

DAMAGE_TO_SEVERITY = {0: 'medium', 1: 'high', 2: 'high', 3: 'low'}
SEVERITY_RANK      = {'low': 0, 'medium': 1, 'high': 2}

def get_severity(label_path):
    if not os.path.exists(label_path) or os.path.getsize(label_path) == 0:
        return 'low'
    best = 'low'
    with open(label_path) as f:
        for line in f:
            parts = line.strip().split()
            if parts:
                sev = DAMAGE_TO_SEVERITY.get(int(parts[0]), 'low')
                if SEVERITY_RANK[sev] > SEVERITY_RANK[best]:
                    best = sev
    return best

by_class = defaultdict(list)
for split in ['train', 'val']:
    images = glob.glob(f'{RDD_ROOT}/{split}/images/*.jpg')
    for img in images:
        label = img.replace('/images/', '/labels/').replace('.jpg', '.txt')
        sev = get_severity(label)
        by_class[sev].append(img)

print("Total collected:")
for cls, imgs in by_class.items():
    print(f"  {cls}: {len(imgs)}")

# Clear old dataset
shutil.rmtree(OUT_DIR, ignore_errors=True)

for split in ['train', 'val']:
    for cls in ['high', 'medium', 'low']:
        os.makedirs(f'{OUT_DIR}/{split}/{cls}', exist_ok=True)

for cls, imgs in by_class.items():
    random.shuffle(imgs)
    imgs = imgs[:1500]
    split_idx = int(len(imgs) * 0.8)
    for img in imgs[:split_idx]:
        shutil.copy(img, f'{OUT_DIR}/train/{cls}/{os.path.basename(img)}')
    for img in imgs[split_idx:]:
        shutil.copy(img, f'{OUT_DIR}/val/{cls}/{os.path.basename(img)}')

print("\nFinal dataset:")
for split in ['train', 'val']:
    for cls in ['high', 'medium', 'low']:
        count = len(os.listdir(f'{OUT_DIR}/{split}/{cls}'))
        print(f"  {split}/{cls}: {count} images")

Total collected:
  medium: 6487
  low: 13722
  high: 12418

Final dataset:
  train/high: 1200 images
  train/medium: 1200 images
  train/low: 1200 images
  val/high: 300 images
  val/medium: 300 images
  val/low: 300 images


In [14]:
import os, time, copy, torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, WeightedRandomSampler
from torchvision import datasets, models, transforms
from torchvision.models import ResNet50_Weights
import numpy as np

def get_transforms():
    train_tf = transforms.Compose([
        transforms.RandomResizedCrop(224, scale=(0.7, 1.0)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomVerticalFlip(p=0.3),
        transforms.RandomRotation(20),
        transforms.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.3, hue=0.1),
        transforms.RandomGrayscale(p=0.05),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
        transforms.RandomErasing(p=0.2),
    ])
    val_tf = transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])
    return train_tf, val_tf

def build_model(num_classes=3):
    model = models.resnet50(weights=ResNet50_Weights.IMAGENET1K_V1)
    for param in model.parameters():
        param.requires_grad = False
    for name, param in model.named_parameters():
        if any(layer in name for layer in ["layer3", "layer4", "fc"]):
            param.requires_grad = True
    num_ftrs = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Dropout(p=0.3),
        nn.Linear(num_ftrs, 256),
        nn.ReLU(),
        nn.Dropout(p=0.2),
        nn.Linear(256, num_classes)
    )
    return model

def train_model(model, dataloaders, criterion, optimizer, scheduler, num_epochs, device, patience=6):
    best_model_wts = copy.deepcopy(model.state_dict())
    best_acc       = 0.0
    epochs_no_improve = 0

    for epoch in range(num_epochs):
        print(f"\nEpoch {epoch+1}/{num_epochs}")
        print("─" * 40)
        for phase in ["train", "val"]:
            model.train() if phase == "train" else model.eval()
            running_loss = 0.0
            running_corrects = 0

            for inputs, labels in dataloaders[phase]:
                inputs, labels = inputs.to(device), labels.to(device)
                optimizer.zero_grad()
                with torch.set_grad_enabled(phase == "train"):
                    outputs = model(inputs)
                    loss = criterion(outputs, labels)
                    _, preds = torch.max(outputs, 1)
                    if phase == "train":
                        loss.backward()
                        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                        optimizer.step()
                running_loss     += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)

            epoch_loss = running_loss / len(dataloaders[phase].dataset)
            epoch_acc  = running_corrects.double() / len(dataloaders[phase].dataset)
            print(f"  {phase.capitalize():5s} — Loss: {epoch_loss:.4f}  Acc: {epoch_acc:.4f} ({epoch_acc*100:.1f}%)")

            if phase == "val":
                scheduler.step(epoch_acc)
                if epoch_acc > best_acc:
                    best_acc = epoch_acc
                    best_model_wts = copy.deepcopy(model.state_dict())
                    torch.save(model.state_dict(), "severity_model.pth")
                    print(f"  ✓ New best! ({epoch_acc*100:.1f}%)")
                    epochs_no_improve = 0
                else:
                    epochs_no_improve += 1
                    print(f"  No improvement {epochs_no_improve}/{patience}")

        if epochs_no_improve >= patience:
            print(f"\nEarly stopping.")
            break
        print(f"  LR: {optimizer.param_groups[-1]['lr']:.6f}")

    print(f"\nBest val accuracy: {best_acc*100:.1f}%")
    model.load_state_dict(best_model_wts)
    return model

# ── Run ──────────────────────────────────────────────────────────
DATA_DIR   = "/content/dataset"
EPOCHS     = 25
BATCH_SIZE = 32
LR         = 0.0003

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

train_tf, val_tf = get_transforms()
train_data = datasets.ImageFolder(os.path.join(DATA_DIR, "train"), transform=train_tf)
val_data   = datasets.ImageFolder(os.path.join(DATA_DIR, "val"),   transform=val_tf)
print(f"Classes: {train_data.classes} | Train: {len(train_data)} | Val: {len(val_data)}")

sampler      = get_weighted_sampler(train_data)
train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, sampler=sampler, num_workers=2)
val_loader   = DataLoader(val_data,   batch_size=BATCH_SIZE, shuffle=False,   num_workers=2)
dataloaders  = {"train": train_loader, "val": val_loader}

model = build_model(num_classes=len(train_data.classes)).to(device)

class_counts  = np.bincount(train_data.targets)
class_weights = torch.tensor(1.0 / class_counts, dtype=torch.float).to(device)
criterion     = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)

optimizer = optim.AdamW([
    {"params": [p for n, p in model.named_parameters() if "layer3" in n], "lr": LR * 0.1},
    {"params": [p for n, p in model.named_parameters() if "layer4" in n], "lr": LR * 0.5},
    {"params": model.fc.parameters(), "lr": LR},
], weight_decay=1e-4)

scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", factor=0.4, patience=3
)

def get_weighted_sampler(dataset):
    class_counts   = np.bincount(dataset.targets)
    weights        = 1.0 / class_counts
    sample_weights = weights[dataset.targets]
    return WeightedRandomSampler(sample_weights, len(sample_weights))

train_model(model, dataloaders, criterion, optimizer, scheduler,
            num_epochs=EPOCHS, device=device, patience=6)

Device: cuda
Classes: ['high', 'low', 'medium'] | Train: 3600 | Val: 900

Epoch 1/25
────────────────────────────────────────
  Train — Loss: 1.0000  Acc: 0.5214 (52.1%)
  Val   — Loss: 0.9943  Acc: 0.5711 (57.1%)
  ✓ New best! (57.1%)
  LR: 0.000300

Epoch 2/25
────────────────────────────────────────
  Train — Loss: 0.9160  Acc: 0.5894 (58.9%)
  Val   — Loss: 0.9095  Acc: 0.6156 (61.6%)
  ✓ New best! (61.6%)
  LR: 0.000300

Epoch 3/25
────────────────────────────────────────
  Train — Loss: 0.8740  Acc: 0.6397 (64.0%)
  Val   — Loss: 0.8976  Acc: 0.6278 (62.8%)
  ✓ New best! (62.8%)
  LR: 0.000300

Epoch 4/25
────────────────────────────────────────
  Train — Loss: 0.8627  Acc: 0.6469 (64.7%)
  Val   — Loss: 0.8938  Acc: 0.6222 (62.2%)
  No improvement 1/6
  LR: 0.000300

Epoch 5/25
────────────────────────────────────────
  Train — Loss: 0.8241  Acc: 0.6686 (66.9%)
  Val   — Loss: 0.9137  Acc: 0.6411 (64.1%)
  ✓ New best! (64.1%)
  LR: 0.000300

Epoch 6/25
──────────────────────────

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (downsample): Sequential(
        (0): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 

In [17]:
# Evaluation
import torch
import torch.nn as nn
from torchvision import models, transforms
from torchvision.models import ResNet50_Weights
from PIL import Image
import os, random

# Load model
CLASS_NAMES = ['high', 'low', 'medium']

model = models.resnet50(weights=None)
model.fc = nn.Sequential(
    nn.Dropout(p=0.3),
    nn.Linear(model.fc.in_features, 256),
    nn.ReLU(),
    nn.Dropout(p=0.2),
    nn.Linear(256, 3)
)
model.load_state_dict(torch.load('severity_model.pth', map_location='cpu'))
model.eval()

transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

def predict(image_path):
    img    = Image.open(image_path).convert('RGB')
    tensor = transform(img).unsqueeze(0)
    with torch.no_grad():
        outputs = model(tensor)
        probs   = torch.softmax(outputs, dim=1)[0]
    scores = {CLASS_NAMES[i]: round(probs[i].item() * 100, 1) for i in range(3)}
    best   = max(scores, key=scores.get)
    return best, scores

# Test 5 random images from each class
DATA_DIR = '/content/dataset/val'
print("=" * 55)
for cls in ['high', 'medium', 'low']:
    print(f"\nActual class: {cls.upper()}")
    print("-" * 55)
    images = os.listdir(f'{DATA_DIR}/{cls}')
    random.shuffle(images)
    correct = 0
    for img_name in images[:5]:
        path = f'{DATA_DIR}/{cls}/{img_name}'
        pred, scores = predict(path)
        status = "✓" if pred == cls else "✗"
        print(f"  {status} Predicted: {pred.upper():6s} | "
              f"HIGH:{scores['high']:5.1f}% "
              f"MED:{scores['medium']:5.1f}% "
              f"LOW:{scores['low']:5.1f}%")
        if pred == cls:
            correct += 1
    print(f"  Accuracy on these 5: {correct}/5")

print("\n" + "=" * 55)
print("Overall spot check complete.")


Actual class: HIGH
-------------------------------------------------------
  ✓ Predicted: HIGH   | HIGH: 73.6% MED:  4.3% LOW: 22.1%
  ✓ Predicted: HIGH   | HIGH: 81.5% MED: 15.8% LOW:  2.7%
  ✗ Predicted: LOW    | HIGH:  2.8% MED:  3.0% LOW: 94.1%
  ✓ Predicted: HIGH   | HIGH: 86.8% MED:  2.5% LOW: 10.6%
  ✗ Predicted: MEDIUM | HIGH: 16.6% MED: 64.9% LOW: 18.5%
  Accuracy on these 5: 3/5

Actual class: MEDIUM
-------------------------------------------------------
  ✓ Predicted: MEDIUM | HIGH:  3.4% MED: 94.7% LOW:  1.9%
  ✗ Predicted: LOW    | HIGH: 13.4% MED: 15.5% LOW: 71.1%
  ✓ Predicted: MEDIUM | HIGH:  1.6% MED: 96.3% LOW:  2.1%
  ✓ Predicted: MEDIUM | HIGH:  1.4% MED: 97.3% LOW:  1.3%
  ✗ Predicted: HIGH   | HIGH: 52.6% MED: 27.5% LOW: 20.0%
  Accuracy on these 5: 3/5

Actual class: LOW
-------------------------------------------------------
  ✓ Predicted: LOW    | HIGH:  4.6% MED:  1.7% LOW: 93.7%
  ✓ Predicted: LOW    | HIGH:  5.4% MED:  4.2% LOW: 90.5%
  ✓ Predicted: LOW   

In [21]:
from google.colab import drive, files
import shutil, os

# Mount Google Drive
drive.mount('/content/drive')

# Create folders
os.makedirs('/content/predictions', exist_ok=True)
os.makedirs('/content/drive/MyDrive/CivicEye/predictions', exist_ok=True)

# Copy all prediction images to Google Drive
for cls in ['high', 'medium', 'low']:
    src = f'/content/predictions/predictions_{cls}.png'
    dst = f'/content/drive/MyDrive/CivicEye/predictions/predictions_{cls}.png'
    if os.path.exists(src):
        shutil.copy(src, dst)
        print(f"Saved to Drive: predictions_{cls}.png")

# Also save the model to Drive
shutil.copy('severity_model.pth', '/content/drive/MyDrive/CivicEye/severity_model_v3_67pct.pth')
print("Saved model to Drive: severity_model_v3_67pct.pth")

# Download all 3 prediction images locally
print("\nDownloading to your PC...")
files.download('/content/predictions/predictions_high.png')
files.download('/content/predictions/predictions_medium.png')
files.download('/content/predictions/predictions_low.png')

print("\nDone! Files saved to:")
print("  Google Drive → MyDrive/CivicEye/predictions/")
print("  Your PC      → Downloads folder")

Mounted at /content/drive
Saved to Drive: predictions_high.png
Saved to Drive: predictions_medium.png
Saved to Drive: predictions_low.png
Saved model to Drive: severity_model_v3_67pct.pth



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


Done! Files saved to:
  Google Drive → MyDrive/CivicEye/predictions/
  Your PC      → Downloads folder
